# Notebook 7: Clustering — KMeans & DBSCAN
## Federal Reserve Interest Rate Prediction

**Objective:** Discover hidden economic regime structure without labels.

- **KMeans:** Centroid-based partitioning — finds k spherical clusters
- **DBSCAN:** Density-based — finds arbitrary-shaped clusters and marks outliers as noise

**Evaluation:** Silhouette Score, Inertia (Elbow Method), Davies-Bouldin Index


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
import pickle


In [ ]:
with open(f"{OUT_PATH}/results/preprocessed_data.pkl", "rb") as f:
    data = pickle.load(f)
X_scaled = data['X_scaled']
X_pca_2 = PCA(n_components=2).fit_transform(X_scaled)
print(f"Data shape: {X_scaled.shape}")


## 1. KMeans — Optimal k Selection

In [ ]:
k_range = range(2, 11)
inertias, sil_scores, db_scores = [], [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    db_scores.append(davies_bouldin_score(X_scaled, labels))

best_k = k_range[np.argmax(sil_scores)]
print(f"Best k (silhouette): {best_k}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].plot(k_range, inertias, 'b-o', linewidth=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Curve', fontweight='bold')

axes[1].plot(k_range, sil_scores, 'r-o', linewidth=2)
axes[1].axvline(best_k, color='green', linestyle='--', label=f'Best k={best_k}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontweight='bold'); axes[1].legend()

axes[2].plot(k_range, db_scores, 'g-o', linewidth=2)
axes[2].set_xlabel('k'); axes[2].set_ylabel('Davies-Bouldin Score')
axes[2].set_title('Davies-Bouldin Index (lower=better)', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km_best.fit_predict(X_scaled)
print(f"KMeans Silhouette: {silhouette_score(X_scaled, km_labels):.4f}")
print(f"KMeans Davies-Bouldin: {davies_bouldin_score(X_scaled, km_labels):.4f}")

fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(X_pca_2[:,0], X_pca_2[:,1], c=km_labels, cmap='tab10', s=20, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Cluster')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'KMeans Clusters (k={best_k}) — PCA 2D Projection', fontweight='bold')
plt.tight_layout(); plt.show()


## 2. DBSCAN — Density-Based Clustering

In [ ]:
# K-distance plot to find optimal eps
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_dist = np.sort(distances[:,-1])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_dist, color='#2196F3')
ax.set_xlabel('Points (sorted)'); ax.set_ylabel('5th Nearest Neighbor Distance')
ax.set_title('DBSCAN: K-Distance Plot (Knee=Best eps)', fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Grid search over eps
eps_grid = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
print(f"{'eps':>8} {'Clusters':>10} {'Noise%':>10} {'Silhouette':>12}")
print("-"*45)
for eps in eps_grid:
    db = DBSCAN(eps=eps, min_samples=5)
    lbl = db.fit_predict(X_scaled)
    n_cls = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_noi = list(lbl).count(-1)
    core  = lbl != -1
    sil   = silhouette_score(X_scaled[core], lbl[core]) if n_cls >= 2 and core.sum() > 1 else -1
    print(f"{eps:>8} {n_cls:>10} {n_noi/len(lbl)*100:>9.1f}% {sil:>12.4f}")


In [ ]:
best_eps = 1.0
db_best  = DBSCAN(eps=best_eps, min_samples=5)
db_labels = db_best.fit_predict(X_scaled)
n_cls = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print(f"DBSCAN: {n_cls} clusters, {list(db_labels).count(-1)} noise points")

fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.tab10(np.linspace(0,1,n_cls+1))
for lbl in set(db_labels):
    mask = db_labels == lbl
    col = 'lightgray' if lbl==-1 else cmap[lbl]
    lab = 'Noise' if lbl==-1 else f'Cluster {lbl}'
    ax.scatter(X_pca_2[mask,0], X_pca_2[mask,1],
               c=[col]*mask.sum(), s=12 if lbl==-1 else 20,
               alpha=0.3 if lbl==-1 else 0.8, label=lab)
ax.set_title(f'DBSCAN (eps={best_eps}) — PCA 2D', fontweight='bold')
ax.legend(); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()


## 3. Cluster Profile Analysis

In [ ]:
import pickle
with open(f"{OUT_PATH}/results/preprocessed_data.pkl", "rb") as f:
    data = pickle.load(f)
df_cluster = data['df'].select_dtypes(include=np.number).iloc[:len(km_labels)]
df_cluster['KMeans'] = km_labels

profile = df_cluster.groupby('KMeans')[['FEDRates','InflationConsumerPrice',
                                         'UnemployemenrRate','GDP']].mean()
print("KMeans Cluster Profiles:")
display(profile.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
profile.T.plot.bar(ax=ax, colormap='Set2')
ax.set_title('KMeans Cluster Economic Profiles', fontweight='bold')
ax.tick_params(axis='x', rotation=0); plt.tight_layout(); plt.show()
